# Руководство администратора
## Приложение поиска фермерских рынков США

**Версия документации:** 1.0  
**Основная реализация:** `Farm_proj_OOP.py`

Документ содержит сведения о развёртывании, проверке, резервном копировании и безопасной эксплуатации текущей CSV-версии приложения.

## Содержание

1. Назначение и архитектура  
2. Состав проекта  
3. Требования к окружению  
4. Развёртывание  
5. Форматы данных  
6. Первичная проверка  
7. Doctest и режим DEBUG  
8. Резервное копирование  
9. Восстановление  
10. Безопасная эксплуатация  
11. Обновление приложения  
12. Диагностика неисправностей  
13. Жизненный цикл и сопровождение  
14. Контрольный список администратора

## 1. Назначение и архитектура

Текущая версия является локальным консольным приложением на Python.

Основные классы ООП-реализации:

- `Market` — представляет один фермерский рынок;
- `MarketManager` — загружает рынки, выполняет поиск, сортировку, расчёт расстояний и удаление;
- `ReviewManager` — загружает, сохраняет, добавляет и удаляет отзывы;
- `FarmMarketApp` — отвечает за взаимодействие с пользователем и обработку команд.

Постоянное хранение данных выполняется в CSV-файлах. Отдельный ZIP-справочник используется для географического поиска.

## 2. Состав проекта

Рекомендуемая структура:

```text
Farm_Market/
├── Farm_proj_OOP.py
├── Farm_proj_REPL.py
├── Export.csv
├── reviews.csv
├── zip_util.py
├── zip_codes_states.csv
└── docs/
    ├── User_Guide.ipynb
    ├── Administrator_Guide.ipynb
    └── screenshots/
```

### Назначение файлов

| Файл | Назначение |
|---|---|
| `Farm_proj_OOP.py` | основная ООП-версия |
| `Farm_proj_REPL.py` | функциональная REPL-версия |
| `Export.csv` | данные фермерских рынков |
| `reviews.csv` | пользовательские отзывы |
| `zip_util.py` | чтение ZIP-справочника |
| `zip_codes_states.csv` | координаты ZIP-кодов |

## 3. Требования к окружению

Минимально необходимы:

- Python 3;
- доступ на чтение файлов проекта;
- доступ на запись `Export.csv` и `reviews.csv` для функций удаления и отзывов;
- достаточно свободного места для исходных CSV и резервных копий.

Программа использует стандартные модули Python `csv`, `math`, `doctest` и локальный модуль `zip_util`. Установка сторонних библиотек для самой программы не требуется.

Для просмотра документации `.ipynb` рекомендуется Jupyter Notebook, JupyterLab или IDE с поддержкой Jupyter.

## 4. Развёртывание

### Шаг 1. Создать каталог приложения

Скопируйте в один каталог основной `.py`-файл и все CSV/служебные файлы.

### Шаг 2. Проверить Python

```bash
python --version
```

### Шаг 3. Проверить наличие файлов

```bash
ls
```

В каталоге должны присутствовать как минимум:

```text
Farm_proj_OOP.py
Export.csv
reviews.csv
zip_util.py
zip_codes_states.csv
```

Если `reviews.csv` отсутствует, приложение способно начать с пустого списка отзывов, но для штатной эксплуатации рекомендуется создать файл через приложение либо заранее подготовить его с корректными заголовками.

### Шаг 4. Запустить

```bash
python Farm_proj_OOP.py
```

После успешного запуска должно появиться меню приложения.

## 5. Форматы данных

### `Export.csv`

Основной файл рынков. Критически важные для работы поля включают:

- `FMID`;
- `MarketName`;
- `city`;
- `State`;
- `street`;
- `zip`;
- `Website`;
- `x`, `y`;
- поля категорий товаров;
- поля способов оплаты.

`FMID` используется как идентификатор рынка.

### `reviews.csv`

Приложение сохраняет четыре поля:

```text
FMID,Name,Review,Rating
```

Файл читается с `utf-8-sig`, что позволяет корректно обрабатывать BOM. При загрузке старое служебное поле `ReviewID` удаляется. При сохранении лишние поля игнорируются.

### `zip_codes_states.csv`

Используется через `zip_util.py` для получения координат ZIP-кодов.

## 6. Первичная проверка после развёртывания

После установки рекомендуется выполнить короткий smoke-test:

1. `show` — открыть первую страницу;
2. `search` — найти `Danville`, `VT`;
3. `zip` — проверить ZIP `05828`;
4. `distance` — выполнить поиск от `05828`;
5. `details` — открыть существующий FMID;
6. `sort` — проверить одну сортировку;
7. `end` — штатно завершить приложение.

Операции `review` и `delete` изменяют данные, поэтому в производственной копии их следует проверять только после создания резервной копии.

## 7. Doctest и режим DEBUG

В исходном коде предусмотрен флаг:

```python
DEBUG = False
```

Для запуска встроенных doctest временно установите:

```python
DEBUG = True
```

и запустите приложение.

Программа вызывает:

```python
doctest.testmod()
```

и выводит количество выполненных тестов и ошибок.

После проверки рекомендуется вернуть:

```python
DEBUG = False
```

Doctest следует запускать после изменений в алгоритмах поиска, сортировки, расчёта расстояний, рейтингов, удаления и обработки ZIP-кодов.

## 8. Резервное копирование

Особенно важны:

- `Export.csv`;
- `reviews.csv`.

Перед обновлением программы или массовым удалением данных создайте копии, например:

```bash
cp Export.csv Export_backup.csv
cp reviews.csv reviews_backup.csv
```

Для регулярной учебной эксплуатации достаточно сохранять резервную копию перед изменениями программы и перед операциями, затрагивающими большое количество данных.

Резервные копии не следует хранить поверх рабочих файлов.

## 9. Восстановление

Если рабочий CSV повреждён или данные были удалены ошибочно:

1. завершите приложение;
2. переименуйте повреждённый файл;
3. восстановите последнюю корректную резервную копию;
4. проверьте заголовки CSV;
5. запустите doctest;
6. выполните smoke-test.

Не рекомендуется восстанавливать файл во время работы приложения.

## 10. Безопасная и стабильная эксплуатация

Для текущего локального учебного приложения основные меры следующие:

- предоставлять право записи только пользователям, которым разрешено менять данные;
- не запускать одновременно несколько экземпляров программы, изменяющих одни и те же CSV-файлы;
- не редактировать `Export.csv` и `reviews.csv` вручную во время работы программы;
- сохранять резервные копии перед изменением структуры CSV;
- не заменять `zip_util.py` или ZIP-справочник непроверенными файлами;
- запускать приложение из каталога проекта, чтобы относительные пути находились корректно;
- после изменения кода выполнять doctest и базовый сценарный тест;
- не удалять заголовки CSV;
- сохранять исходные файлы в предусмотренной кодировке.

Текущая версия не является сетевым сервисом и не имеет системы учётных записей. Поэтому сетевые настройки, TLS и управление серверными пользователями к этой версии не применяются.

## 11. Обновление приложения

Рекомендуемый порядок:

1. создать резервную копию данных;
2. сохранить предыдущую рабочую версию `.py`;
3. внести изменения в отдельную копию;
4. проверить синтаксис;
5. выполнить doctest;
6. выполнить smoke-test;
7. только после успешной проверки заменить рабочую версию.

Проверка синтаксиса:

```bash
python -m py_compile Farm_proj_OOP.py
```

При изменении формата CSV необходимо отдельно проверить совместимость старых данных.

## 12. Диагностика неисправностей

| Проблема | Возможная причина | Действие |
|---|---|---|
| `FileNotFoundError` | отсутствует файл или запуск из другой папки | проверить каталог запуска и состав проекта |
| не находится ZIP | ZIP отсутствует в справочочнике | проверить `zip_codes_states.csv` |
| ошибка полей CSV | изменены заголовки | восстановить штатную структуру |
| проблемы с `FMID` после внешнего редактирования | BOM/кодировка или изменён заголовок | проверить первый заголовок и кодировку |
| отзыв не сохраняется | нет права записи | проверить права на `reviews.csv` |
| рынок не удаляется | нет права записи или повреждён CSV | проверить права и резервную копию |
| doctest завершился с ошибками | изменилось поведение функции или тестовые данные | изучить конкретный failed example |

## 13. Жизненный цикл и сопровождение

Для учебного проекта достаточно следующего цикла:

**Разработка → тестирование → резервное копирование → выпуск версии → эксплуатация → исправление ошибок → повторное тестирование.**

После каждого функционального изменения необходимо:

- проверить соответствующие doctest;
- проверить пользовательский сценарий вручную;
- убедиться, что существующие CSV читаются;
- обновить документацию, если изменились команды или поведение.

При переходе в дальнейшем на PostgreSQL руководство администратора потребуется дополнить установкой СУБД, созданием базы и пользователя, параметрами подключения, миграциями, резервным копированием БД и управлением правами.

## 14. Контрольный список администратора

Перед передачей приложения пользователю:

- [ ] Python установлен и запускается;
- [ ] все файлы проекта находятся в нужном каталоге;
- [ ] `Export.csv` читается;
- [ ] `reviews.csv` доступен на запись;
- [ ] ZIP-справочник доступен;
- [ ] doctest проходят без ошибок;
- [ ] `show`, `search`, `zip`, `distance`, `details`, `sort` проверены;
- [ ] создана резервная копия рабочих данных;
- [ ] `DEBUG` возвращён в `False`;
- [ ] документация соответствует текущей версии программы.